<a href="https://colab.research.google.com/github/Surya-devi289/sseccse/blob/main/Alram_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required library (Run once in Colab)
!pip install gTTS nltk pytz

import re
import time
import os
from datetime import datetime, timedelta
import pytz
from gtts import gTTS # Use gTTS for text-to-speech
from IPython.display import Audio, display

# ------------------ ELEVENLABS SETUP (REMOVED/COMMENTED OUT) ------------------
# No API key needed for gTTS
# os.environ["ELEVENLABS_API_KEY"] = "YOUR_API_KEY"
# client = ElevenLabs(
#     api_key=os.getenv("ELEVENLABS_API_KEY")
# )
# VOICE_ID = "56AoDkrOh6qfVPDXZ7Pt"

# ------------------ NLP CHATBOT ------------------

def chatbot_response(text):

    text = text.lower()

    # Greeting intent
    if any(word in text for word in ["hi", "hello", "hey"]):
        return "Hello! I can set alarms for you."

    # Name intent
    elif "your name" in text:
        return "I am an AI Alarm Chatbot."

    # Alarm intent
    elif "alarm" in text:
        return "Please enter the alarm time in HH:MM format."

    # Exit intent
    elif text == "bye":
        return "Goodbye!"

    else:
        return "Sorry, I didn't understand."

# ------------------ EXTRACT ALARM TIME ------------------

def extract_time(user_text):

    # Modified pattern to support both HH.MM and HH:MM formats
    pattern = r'(\d{1,2})[.:](\d{2})'

    match = re.search(pattern, user_text)

    if match:
        hour = match.group(1)
        minute = match.group(2)
        return f"{hour.zfill(2)}:{minute}"

    return None

# ------------------ GENERATE VOICE (USING gTTS) ------------------

def speak_message(message):

    tts = gTTS(text=message, lang='en')
    filename = "alarm.mp3"
    tts.save(filename)
    display(Audio(filename, autoplay=True))

# ------------------ MAIN PROGRAM ------------------

print("AI Alarm Chatbot")
print("Type 'bye' to exit")

alarm_datetime = None

# Define the Indian Standard Time (IST) timezone
ist = pytz.timezone('Asia/Kolkata')

while True:

    user = input("You: ")

    if user.lower() == "bye":
        print("Chatbot: Goodbye!")
        break

    response = chatbot_response(user)
    print("Chatbot:", response)

    # Check whether user wants to set an alarm
    if "alarm" in user.lower():

        time_input = input("Enter alarm time (HH:MM or HH.MM): ")

        alarm_time_str = extract_time(time_input)

        if alarm_time_str:
            # Get current date in IST
            now_ist = datetime.now(ist)
            # Combine current date with user-provided time and set to IST
            alarm_hour, alarm_minute = map(int, alarm_time_str.split(':'))
            alarm_datetime = now_ist.replace(hour=alarm_hour, minute=alarm_minute, second=0, microsecond=0)

            # If the alarm time is in the past, assume it's for the next day
            if alarm_datetime < now_ist:
                alarm_datetime += timedelta(days=1)

            print(f"Chatbot: Alarm set for {alarm_datetime.strftime('%H:%M')} IST")

            # Wait for alarm
            while True:

                current_time_ist = datetime.now(ist)

                print("Current Time (IST):", current_time_ist.strftime("%H:%M:%S"))

                # Compare timezone-aware datetime objects
                if current_time_ist >= alarm_datetime:

                    print("⏰ Alarm Ringing!")

                    message = "Good Morning! It's time to wake up."

                    speak_message(message)

                    break

                time.sleep(30)

        else:
            print("Invalid time format. Please use HH:MM or HH.MM.")

AI Alarm Chatbot
Type 'bye' to exit
Chatbot: Hello! I can set alarms for you.
Chatbot: Please enter the alarm time in HH:MM format.
Chatbot: Alarm set for 19:29 IST
Current Time (IST): 19:28:53
Current Time (IST): 19:29:23
⏰ Alarm Ringing!
